In [1]:
from dotenv import load_dotenv
load_dotenv()
from rag_helper import RAGBase
from openai import OpenAI


In [2]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [3]:
openai_client = OpenAI()

assistant = RAGBase(
    sqlite_index,
    llm_client=openai_client,
)



In [4]:
answer = assistant.rag("How do I run olama in locally?")
print(answer)

To run Ollama locally:

1. Install Ollama from: https://ollama.com/download  
   - macOS: install the `.pkg`
   - Windows: install the `.msi`
   - Linux: run:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. Start a model in the terminal:
```bash
ollama run llama3
```

3. To check that the local server is running, try:
```bash
curl http://localhost:11434
```

If you want, you can also install the Python client:
```bash
pip install ollama
```

I don't know.


In [5]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [6]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join late enrollment eligibility"}', call_id='call_i4RLqbRaHbqmoOTfbFKlbtZB', name='search', type='function_call', id='fc_0708b208b848349d006a29c821e09481a296912e621111d179', namespace=None, status='completed')]

In [7]:
call=response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join late enrollment eligibility"}', call_id='call_i4RLqbRaHbqmoOTfbFKlbtZB', name='search', type='function_call', id='fc_0708b208b848349d006a29c821e09481a296912e621111d179', namespace=None, status='completed')

In [8]:
import json
args=json.loads(call.arguments)
results=search(**args)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have r

In [9]:
results_json=json.dumps(results,indent=2)
print(results_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."
  },
  {
    "id": "977bf7786c",
    "course": "llm-zoomcamp",
    "section": "General Course

In [10]:
messages.append(call)
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join late enrollment eligibility"}', call_id='call_i4RLqbRaHbqmoOTfbFKlbtZB', name='search', type='function_call', id='fc_0708b208b848349d006a29c821e09481a296912e621111d179', namespace=None, status='completed')]

In [11]:
function_call_output={
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": results_json,
}
messages.append(function_call_output)

In [12]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)



In [13]:
print(response.output_text)

Yes — you can still join.

If you want a certificate, make sure to submit your project while submissions are still open.


In [14]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(650, 29)

In [15]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [16]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [17]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)


In [19]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [20]:
messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join the course enrollment discovered course can I join late registration FAQ"}


In [21]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join the course enrollment discovered course can I join late registration FAQ"}', call_id='call_Nt3IIhwyrCqxgfATPxlQoK2j', name='search', type='function_call', id='fc_073c2dc97af0c696006a29c82475f481a19454fc2dcdc2cf28', namespace=None, status='completed')]

In [22]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join late enrollment registration"}
function_call: search {"query":"course FAQ enrollment late join discovered course"}
function_call: search {"query":"can I join the course after it started FAQ"}


In [23]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still being accepted. Also, if you want a certificate, you need to participate with a live cohort rather than only following it fully self-paced.

If you’d like, I can also help with what to do next after joining. Are there other areas you want to explore?


In [24]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [25]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join the course late enrollment discovered the course can I join FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, you’ll need to submit your project while the course is still accepting submissions.

If you’re just learning for the content, you can start anytime. Would you like help with anything else about the course?


'Yes — you can still join the course.\n\nIf you want a certificate, you’ll need to submit your project while the course is still accepting submissions.\n\nIf you’re just learning for the content, you can start anytime. Would you like help with anything else about the course?'

In [26]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [27]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [28]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

agent_tools = Tools()
agent_tools.add_tool(search)

In [29]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [30]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [31]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [32]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [33]:
result.cost

CostInfo(input_cost=Decimal('0.002391'), output_cost=Decimal('0.001665'), total_cost=Decimal('0.004056'))